In [0]:
from pyspark.sql.functions import *

# 2) Visualização completa da base

In [0]:
%sql
SELECT *
FROM projetos_jr.camada_prata.pr_transacoes_bancarias

#1) Produto de dados 1: Visão financeira dos cliente

In [0]:
df_atividade_cliente = spark.sql(
    """
    SELECT nome_cliente, count(*) as qtd_transacoes, sum(valor_transacao) as total_movimentado, avg((valor_transacao)) as ticket_medio, max(data_transacao) as ultima_transacao
    FROM projetos_jr.camada_prata.pr_transacoes_bancarias
    GROUP BY nome_cliente 
    """
)

display(df_atividade_cliente)

# Produto de dados 2: Indicadores gerais do banco

In [0]:
df_visao_banco = spark.sql(
    """
    SELECT count(nome_cliente) as qtd_clientes, sum(valor_transacao) as volume_movimentado, count(*) as qnt_transacoes, avg(valor_transacao) as media_transacoes,
    count(CASE WHEN tipo_transacao = 'PIX' THEN 1 END) as qnt_transcoes_pix,
    count(CASE WHEN status_transacao = 'Cancelada' THEN 1 END) as qnt_transacoes_canceladas
    FROM projetos_jr.camada_prata.pr_transacoes_bancarias
    """
)

display(df_visao_banco)

# Produto de dados 3: Monitoramento de risco

In [0]:
df_monitoramento_risco = spark.sql(
    """
    SELECT 
        nome_cliente,
        count(CASE WHEN status_transacao = 'Cancelada' THEN 1 END) as qnt_transacoes_canceladas,
        sum(CASE WHEN valor_transacao > (SELECT avg(valor_transacao) * 2 FROM projetos_jr.camada_prata.pr_transacoes_bancarias) THEN 1 ELSE 0 END) as qnt_valores_3x_acima_media
    FROM projetos_jr.camada_prata.pr_transacoes_bancarias
    GROUP BY nome_cliente
    """
)

In [0]:
display(df_monitoramento_risco)

# Salvando dentro do ambiente analítico

In [0]:
catalog_name = 'projetos_jr'
database_name = 'camada_ouro'
table_name = 'v_finan_cli'

df_atividade_cliente.write.mode("overwrite").saveAsTable(f'{catalog_name}.{database_name}.{table_name}')

In [0]:
catalog_name = 'projetos_jr'
database_name = 'camada_ouro'
table_name = 'v_gral_banco'

df_visao_banco.write.mode("overwrite").saveAsTable(f'{catalog_name}.{database_name}.{table_name}')

In [0]:
catalog_name = 'projetos_jr'
database_name = 'camada_ouro'
table_name = 'monit_risco'

df_monitoramento_risco.write.mode("overwrite").saveAsTable(f'{catalog_name}.{database_name}.{table_name}')